# 08 — Agents & Tool Calling

`🔴 Advanced`

---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MusaIslamFahad/python-for-ai-engineers/blob/main/07_AI_Engineering/08_agents_and_tools.ipynb)

## 📌 What is it?
An AI agent uses an LLM to reason, decide which tools to use, execute them, observe results, and repeat until the task is complete.

## 🧠 Why AI Engineers need this
Tool calling (function calling) is the mechanism behind AI assistants that can search the web, run code, query databases, and use APIs. Understanding this lets you build autonomous AI systems.

In [ ]:
import json

# ── TOOL DEFINITION (OpenAI format) ──
tools = [
    {
        "type": "function",
        "function": {
            "name": "search_documents",
            "description": "Search the document database for relevant information",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "The search query"
                    },
                    "top_k": {
                        "type": "integer",
                        "description": "Number of results to return (default: 3)",
                        "default": 3
                    }
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "run_python_code",
            "description": "Execute Python code and return the output",
            "parameters": {
                "type": "object",
                "properties": {
                    "code": {
                        "type": "string",
                        "description": "Python code to execute"
                    }
                },
                "required": ["code"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_current_time",
            "description": "Get the current date and time",
            "parameters": {"type": "object", "properties": {}, "required": []}
        }
    }
]

print("Tools defined:")
for tool in tools:
    fn = tool["function"]
    print(f"  🔧 {fn['name']}: {fn['description']}")

In [ ]:
import json
from datetime import datetime

# ── TOOL IMPLEMENTATIONS ──
def search_documents(query: str, top_k: int = 3) -> str:
    """Mock document search."""
    mock_results = [
        f"[Doc {i+1}] Relevant info about '{query}': chunk {i+1}"
        for i in range(top_k)
    ]
    return json.dumps({"results": mock_results, "query": query})

def run_python_code(code: str) -> str:
    """Safely execute Python code in a sandbox."""
    import io, contextlib
    stdout = io.StringIO()
    try:
        with contextlib.redirect_stdout(stdout):
            exec(code, {"__builtins__": __builtins__})
        return json.dumps({"output": stdout.getvalue(), "error": None})
    except Exception as e:
        return json.dumps({"output": None, "error": str(e)})

def get_current_time() -> str:
    return json.dumps({"datetime": datetime.now().isoformat(), "timezone": "UTC"})

TOOL_MAP = {
    "search_documents": search_documents,
    "run_python_code":  run_python_code,
    "get_current_time": get_current_time,
}

# Test tools
print("Testing tools:")
print(search_documents("RAG pipeline", top_k=2))
print(run_python_code("result = sum(range(100))\nprint(f'Sum: {result}')"))
print(get_current_time())

In [ ]:
import json

# ── REACT AGENT LOOP (Reason + Act) ──
class SimpleAgent:
    """ReAct-style agent: Reason → Act → Observe → Repeat."""
    
    def __init__(self, tool_map: dict, max_steps: int = 5):
        self.tool_map = tool_map
        self.max_steps = max_steps
        self.history = []
    
    def execute_tool(self, tool_name: str, tool_args: dict) -> str:
        """Execute a tool and return its result."""
        if tool_name not in self.tool_map:
            return json.dumps({"error": f"Unknown tool: {tool_name}"})
        fn = self.tool_map[tool_name]
        return fn(**tool_args)
    
    def run(self, user_query: str) -> str:
        """Run the agent loop (simulation without real LLM)."""
        print(f"🧠 Agent received: {user_query}\n")
        
        # Simulated agent reasoning (in production, LLM decides)
        plan = [
            {"reason": "I should search for relevant docs first",
             "action": "search_documents", "args": {"query": user_query}},
            {"reason": "Let me calculate something to demonstrate code execution",
             "action": "run_python_code", "args": {"code": "print(42 ** 2)"}},
            {"reason": "I have all the info I need to answer",
             "action": None, "args": None}
        ]
        
        for step, item in enumerate(plan, 1):
            print(f"Step {step}/{len(plan)}: {item['reason']}")
            
            if item["action"] is None:
                print("✅ Task complete. Generating final answer...")
                return "Based on the retrieved documents and calculations, here is the answer..."
            
            print(f"  🔧 Calling: {item['action']}({item['args']})")
            result = self.execute_tool(item["action"], item["args"])
            observation = json.loads(result)
            print(f"  👁️  Observation: {str(observation)[:80]}...")
            print()
        
        return "Could not complete task within step limit."

agent = SimpleAgent(TOOL_MAP)
final_answer = agent.run("What documents do you have about RAG?")
print(f"\nFinal Answer: {final_answer}")

In [ ]:
# ── ANTHROPIC TOOL CALLING (Real API pattern) ──
anthropic_tools_code = '''
import anthropic
import json

client = anthropic.Anthropic()

tools = [{
    "name": "search_web",
    "description": "Search the web for up-to-date information",
    "input_schema": {
        "type": "object",
        "properties": {
            "query": {"type": "string", "description": "Search query"}
        },
        "required": ["query"]
    }
}]

def run_agent(user_message: str):
    messages = [{"role": "user", "content": user_message}]
    
    while True:
        response = client.messages.create(
            model="claude-3-5-sonnet-20241022",
            max_tokens=4096,
            tools=tools,
            messages=messages
        )
        
        # Add assistant response to history
        messages.append({"role": "assistant", "content": response.content})
        
        if response.stop_reason == "end_turn":
            # Final response — extract text
            for block in response.content:
                if hasattr(block, "text"):
                    return block.text
        
        elif response.stop_reason == "tool_use":
            # Execute tools and add results
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    result = execute_tool(block.name, block.input)
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": result
                    })
            messages.append({"role": "user", "content": tool_results})
'''
print("Anthropic tool calling loop pattern:")
print(anthropic_tools_code)

## ⭐ You've completed the AI Engineering section!

You now have the core Python skills to build production AI systems:
- REST API calls and error handling
- LLM SDK usage (OpenAI & Anthropic)
- Prompt engineering and output parsing
- Embeddings and semantic search
- Vector databases
- LangChain chains
- Full RAG pipelines
- Agents and tool calling

## 🔗 What's Next?
[08 Projects →](../08_Projects/)